In [2]:
# ============================================================
# CELL 1: IMPORTS & CONFIG
# ============================================================
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
import os

# ============================================================
# STATION FILES
# ============================================================
station_files = {
    "Cibadak":             r"D:\UNIVERSITAS INDONESIA\Bimbingan 2026\Icha\aws_cibadak.csv",
    "Cibaliung":           r"D:\UNIVERSITAS INDONESIA\Bimbingan 2026\Icha\aws_cibaliung.csv",
    "Cileles":             r"D:\UNIVERSITAS INDONESIA\Bimbingan 2026\Icha\aws_cileles (1).csv",
    "Leuwi Dimar":         r"D:\UNIVERSITAS INDONESIA\Bimbingan 2026\Icha\aws_leuwidamar.csv",
    "Singamerta":          r"D:\UNIVERSITAS INDONESIA\Bimbingan 2026\Icha\aws_singamerta.csv",
    "Kebun Bibit Cibubur": r"D:\UNIVERSITAS INDONESIA\Bimbingan 2026\Icha\aws_kebunbibitcibubur.csv",
    "BSD Serpong":         r"D:\UNIVERSITAS INDONESIA\Bimbingan 2026\Icha\aws_bsdserpong (1).csv",
    "Golf Modern":         r"D:\UNIVERSITAS INDONESIA\Bimbingan 2026\Icha\aws_golfmodern.csv",
    "PIK":                 r"D:\UNIVERSITAS INDONESIA\Bimbingan 2026\Icha\aws_pik (3).csv",
    "TMII":                r"D:\UNIVERSITAS INDONESIA\Bimbingan 2026\Icha\aws_tmii.csv",
    "Staklim Banten":      r"D:\UNIVERSITAS INDONESIA\Bimbingan 2026\Icha\aws_staklimbanten (9).csv",
}

OUTPUT_DIR    = "output_pipeline_v3"
WINDOW_SIZE   = "1h"
GAP_THRESHOLD = 1   # menit, gap > ini dianggap missing data dalam window

RULE_DIR = os.path.join(OUTPUT_DIR, "01_rule_based_")
ML_DIR   = os.path.join(OUTPUT_DIR, "02_ml_")
for d in [OUTPUT_DIR, RULE_DIR, ML_DIR]:
    os.makedirs(d, exist_ok=True)

# ============================================================
# RULE THRESHOLDS
# ============================================================
RULE_CONFIG = {
    # Value-based rules
    'std_stuck':        0.1,
    'temp_overheat':    40.0,
    'saturation_max':   -34.0,  # temp_max < ini → saturation pattern
    'saturation_min':   -45.0,  # temp_min > ini → bukan extreme noise
    
    # Gap rules
    'min_valid':        50,    # sensor valid count threshold
    'major_gap_min':    15,    # gap continuous > ini → major
}

# ============================================================
# EXCLUDE_COLS — kolom yang TIDAK masuk ML training
# ============================================================
EXCLUDE_COLS = [
    # Raw sensor (mentah)
    "temp", "rh", "sr", "ws", "press", "rr",
    # Metadata
    "time_diff", "gap_flag", "block_id",
    "num_data", "sampling_rate",
    "window_start", "window_end",
    "window_start_utc", "window_end_utc",
    "WIB_start_time", "WIB_end_time",
    "nama_site", "hour_index",
    # Sensor at_end (raw, gak dipakai ML)
    "temp_at_end", "rh_at_end", "sr_at_end",
    "rr_at_end", "ws_at_end", "press_at_end",
    # Gap stats (untuk rule, bukan ML feature)
    "n_gaps", "max_gap_min",
    "n_rows", "n_valid_rh", "n_valid_press", "n_valid_sr",
    "max_gap_temp", "max_gap_rh", "max_gap_press",
    # Rule output
    "rule_flag", "final_label", "is_rule_flagged",
    # Cross-sensor context (Excel info, bukan ML)
    "rr_sum", "rr_max_rate",
    "sr_mean", "sr_max", "sr_std",
    "rh_mean", "rh_min", "rh_max", "rh_std",
]

DROP_RAW_COLS = ["id", "sn", "ws_max", "wd", "temp_max", "temp_min",
                 "sr_max", "log_temp", "batt"]

print(f"Output dir   : {OUTPUT_DIR}")
print(f"Rule dir     : {RULE_DIR}")
print(f"ML dir       : {ML_DIR}")

Output dir   : output_pipeline_v3
Rule dir     : output_pipeline_v3\01_rule_based_
ML dir       : output_pipeline_v3\02_ml_


In [3]:
# ============================================================
# CELL 2: PREPROCESSING
# ============================================================
def preprocess(file_path):
    df = pd.read_csv(file_path)

    mask_valid = (
        (df['date'].astype(str).str.lower() != 'undefined') &
        (df['time'].astype(str).str.lower() != 'undefined')
    )
    df = df[mask_valid].copy()

    df['datetime'] = pd.to_datetime(df['date'] + ' ' + df['time'], dayfirst=True, errors='coerce')
    df = df.dropna(subset=['datetime'])

    df['datetime'] = (
        df['datetime']
        .dt.tz_localize('UTC')
        .dt.tz_convert('Asia/Jakarta')
        .dt.tz_localize(None)
    )
    df = df.set_index('datetime').sort_index()

    df = df[(df.index.year >= 2023) & (df.index.year <= 2025)]
    df = df.drop(columns=[c for c in DROP_RAW_COLS if c in df.columns])

    df["time_diff"] = df.index.to_series().diff().dt.total_seconds() / 60.0
    print(f"  [INFO] Gap > {GAP_THRESHOLD} menit: {(df['time_diff'] > GAP_THRESHOLD).sum()}")

    # Handle duplikat (Opsi C: keep first)
    n_dup = df.index.duplicated().sum()
    print(f"  [INFO] Duplikat ditemukan: {n_dup}")
    if n_dup > 0:
        # Cek karakteristik duplikat (untuk reporting)
        duplicates = df[df.index.duplicated(keep=False)]
        groups = duplicates.groupby(duplicates.index)
        n_lossless = sum(g['temp'].nunique() <= 1 for _, g in groups if 'temp' in g)
        n_conflict = sum(g['temp'].nunique() > 1 for _, g in groups if 'temp' in g)
        print(f"         - Lossless (nilai identik): {n_lossless}")
        print(f"         - Konflik (nilai berbeda):  {n_conflict}")
        
        # keep first
        df = df[~df.index.duplicated(keep='first')]
        print(f"  [INFO] Sisa duplikat setelah keep-first: {df.index.duplicated().sum()}")

    return df

In [4]:
# ============================================================
# CELL 3: FEATURE EXTRACTION
# ============================================================
def extract_window_features(grp, sensor_cols, all_temp_series):
    """Extract features dari 1 window 1 jam.
    
    Fitur ML:
      - Distribution: temp_mean, temp_std, temp_min, temp_max, temp_range,
                      temp_skew, temp_kurt, temp_cv
      - Rate of change: delta_T_mean, delta_T_std, delta_T_abs_max,
                        delta_T_per_minute_max
      - Persistence: consecutive_identical, roll_1h_unique_ratio
      - Statistical: temp_zscore_max
      - Diurnal: hour_sin, hour_cos, month_sin, month_cos (pakai window_start.hour)
    
    Cross-sensor context (untuk Excel, di-EXCLUDE dari ML):
      - rr_sum, rr_max_rate, sr_mean, sr_max, sr_std,
        rh_mean, rh_min, rh_max, rh_std
    """
    temp = grp["temp"].dropna()
    n    = len(temp)
    if n < 2:
        return None
    feat = {}

    # Window metadata
    feat["num_data"] = n

    # Sensor at_end (untuk Excel output, di-EXCLUDE dari ML)
    for col in sensor_cols:
        if col in grp.columns:
            feat[f"{col}_at_end"] = grp[col].iloc[-1]

    # ============================================================
    # CROSS-SENSOR CONTEXT (untuk Excel, di-EXCLUDE dari ML)
    # ============================================================
    
    # Rainfall — handle cumulative dengan reset midnight via diff > 0 only
    if "rr" in grp.columns:
        rr = grp["rr"].dropna()
        if len(rr) >= 2:
            diffs = rr.diff().dropna()
            pos = diffs[diffs > 0]
            feat["rr_sum"]      = float(pos.sum()) if len(pos) > 0 else 0.0
            feat["rr_max_rate"] = float(pos.max()) if len(pos) > 0 else 0.0
        else:
            feat["rr_sum"]      = np.nan
            feat["rr_max_rate"] = np.nan
    else:
        feat["rr_sum"]      = np.nan
        feat["rr_max_rate"] = np.nan

    # Solar radiation
    if "sr" in grp.columns:
        sr = grp["sr"].dropna()
        if len(sr) >= 2:
            feat["sr_mean"] = float(sr.mean())
            feat["sr_max"]  = float(sr.max())
            feat["sr_std"]  = float(sr.std())
        elif len(sr) == 1:
            feat["sr_mean"] = float(sr.iloc[0])
            feat["sr_max"]  = float(sr.iloc[0])
            feat["sr_std"]  = np.nan
        else:
            feat["sr_mean"] = np.nan
            feat["sr_max"]  = np.nan
            feat["sr_std"]  = np.nan
    else:
        feat["sr_mean"] = np.nan
        feat["sr_max"]  = np.nan
        feat["sr_std"]  = np.nan

    # Humidity
    if "rh" in grp.columns:
        rh = grp["rh"].dropna()
        if len(rh) >= 2:
            feat["rh_mean"] = float(rh.mean())
            feat["rh_min"]  = float(rh.min())
            feat["rh_max"]  = float(rh.max())
            feat["rh_std"]  = float(rh.std())
        elif len(rh) == 1:
            feat["rh_mean"] = float(rh.iloc[0])
            feat["rh_min"]  = float(rh.iloc[0])
            feat["rh_max"]  = float(rh.iloc[0])
            feat["rh_std"]  = np.nan
        else:
            feat["rh_mean"] = np.nan
            feat["rh_min"]  = np.nan
            feat["rh_max"]  = np.nan
            feat["rh_std"]  = np.nan
    else:
        feat["rh_mean"] = np.nan
        feat["rh_min"]  = np.nan
        feat["rh_max"]  = np.nan
        feat["rh_std"]  = np.nan

    # ============================================================
    # FITUR ML (yang dipakai untuk training)
    # ============================================================
    
    # 1. Distribution stats
    feat["temp_mean"]  = temp.mean()
    feat["temp_std"]   = temp.std()
    feat["temp_min"]   = temp.min()
    feat["temp_max"]   = temp.max()
    feat["temp_range"] = temp.max() - temp.min()
    feat["temp_skew"]  = temp.skew()
    feat["temp_kurt"]  = temp.kurt()
    feat["temp_cv"]    = temp.std() / abs(temp.mean()) if temp.mean() != 0 else np.nan

    # 2. Rate of change
    delta = temp.diff()
    feat["delta_T_mean"]    = delta.mean()
    feat["delta_T_std"]     = delta.std()
    feat["delta_T_abs_max"] = delta.abs().max()

    if "time_diff" in grp.columns:
        time_diff = grp["time_diff"].replace(0, np.nan)
        delta_per_min = (grp["temp"].diff() / time_diff).dropna()
        feat["delta_T_per_minute_max"] = delta_per_min.abs().max()
    else:
        feat["delta_T_per_minute_max"] = np.nan

    # 3. Diurnal — pakai window_start.hour (konsisten dengan WIB_start_time)
    # Window 14:00-15:00 → hour = 14
    end_time = grp.index[-1]
    window_start = end_time - pd.Timedelta('1h')
    
    hour     = window_start.hour
    month    = window_start.month
    feat["hour_sin"]  = np.sin(2 * np.pi * hour  / 24.0)
    feat["hour_cos"]  = np.cos(2 * np.pi * hour  / 24.0)
    feat["month_sin"] = np.sin(2 * np.pi * month / 12.0)
    feat["month_cos"] = np.cos(2 * np.pi * month / 12.0)

    # 4. Persistence
    stuck_tolerance = 0.1
    is_same = delta.abs() <= stuck_tolerance
    groups_persist  = (~is_same).cumsum()
    feat["consecutive_identical"] = is_same.groupby(groups_persist).cumsum().max()
    feat["1h_unique_ratio"]  = temp.nunique() / n

    # 5. Statistical — temp_zscore_max only
    # Max dari z-score absolute setiap titik dalam window. Catch spike & drop di mana saja.
    if temp.std() > 0:
        zscore_all = (temp - temp.mean()).abs() / temp.std()
        feat["temp_zscore_max"] = float(zscore_all.max())
    else:
        feat["temp_zscore_max"] = np.nan

    return feat

In [5]:
# ============================================================
# CELL 4: BUILD WINDOW DATASET
# ============================================================

def compute_per_sensor_gap(grp, sensor):
    """Hitung max gap continuous di mana sensor = NaN dalam window."""
    if sensor not in grp.columns:
        return 0.0
    is_nan = grp[sensor].isna()
    if not is_nan.any():
        return 0.0
    timestamps = grp.index
    max_gap = 0.0
    in_run = False
    run_start = None
    for i, (ts, nan_flag) in enumerate(zip(timestamps, is_nan)):
        if nan_flag and not in_run:
            in_run = True
            run_start = ts
        elif not nan_flag and in_run:
            duration = (timestamps[i-1] - run_start).total_seconds() / 60.0
            max_gap = max(max_gap, duration)
            in_run = False
    if in_run:
        duration = (timestamps[-1] - run_start).total_seconds() / 60.0
        max_gap = max(max_gap, duration)
    return max_gap


def build_window_dataset(df, station_name):
    """Resample ke 1h non-overlapping window."""
    sensor_cols = [c for c in ["temp", "rh", "sr", "rr", "ws", "press"] if c in df.columns]
    rows = []
    all_temp_series = df["temp"].dropna()
    groups = df.groupby(pd.Grouper(freq=WINDOW_SIZE, label="right", closed="right"))

    for window_end, grp in groups:
        if len(grp) < 1:
            continue

        feat = extract_window_features(grp, sensor_cols, all_temp_series)
        if feat is None:
            continue

        # Gap metadata
        feat["n_rows"] = len(grp)
        feat["sampling_rate"] = feat["n_rows"] / 60.0
        for s in ["rh", "press", "sr"]:
            feat[f"n_valid_{s}"] = int(grp[s].notna().sum()) if s in grp.columns else 0
        # Per-sensor max continuous gap (untuk gap rules)
        for s in ["temp", "rh", "press"]:
            feat[f"max_gap_{s}"] = compute_per_sensor_gap(grp, s)

        if "time_diff" in grp.columns:
            gaps_in_window = grp["time_diff"].dropna()
            feat["n_gaps"]      = int((gaps_in_window > GAP_THRESHOLD).sum())
            feat["max_gap_min"] = float(gaps_in_window.max()) if len(gaps_in_window) > 0 else 0.0
        else:
            feat["n_gaps"]      = 0
            feat["max_gap_min"] = 0.0

        # Window labels
        window_start = window_end - pd.Timedelta(WINDOW_SIZE)
        # WIB labels (jadi kolom proper di Excel, replace datetime index)
        feat["WIB_start_time"]   = window_start.strftime('%Y-%m-%d %H:%M')
        feat["WIB_end_time"]     = window_end.strftime('%Y-%m-%d %H:%M')
        feat["window_start_utc"] = (window_start - pd.Timedelta(hours=7)).strftime("%Y-%m-%d %H:%M")
        feat["window_end_utc"]   = (window_end   - pd.Timedelta(hours=7)).strftime("%Y-%m-%d %H:%M")
        # Datetime objects untuk internal use (plotting, joining)
        feat["window_start"]     = window_start
        feat["window_end"]       = window_end
        feat["hour_index"]       = window_start.hour  # konsisten dengan temporal encoding
        feat["nama_site"]        = station_name

        rows.append(feat)

    df_windows = pd.DataFrame(rows).set_index("window_end")
    df_windows.index.name = "datetime"
    df_windows = df_windows.sort_index()
    
    print(f"  [INFO] Total windows: {len(df_windows)}")
    return df_windows

In [6]:
# ============================================================
# CELL 5: RULE PREFILTER
# ============================================================
# Mapping rule_flag → final_label:
#   SENSOR_FAILURE          : SATURATION_STUCK (full saturasi -39)
#   SENSOR_ANOMALY          : MEAN_NEGATIVE_FLAKY, HAS_NEGATIVE, HARD_OVERHEAT
#   GAP_FAULT               : comm_few_rows_multisensor, comm_major_gap_multisensor
#   SENSOR_ANOMALY_GAP_DATA : comm_few_rows_temp, comm_major_gap_temp
#   SENSOR_FAILURE_WITH_GAP : SENSOR_FAILURE + has gap issue
#   SENSOR_ANOMALY_WITH_GAP : SENSOR_ANOMALY + has gap issue
# ============================================================

def apply_rule_prefilter(df_windows, cfg=None):
    """Apply rule prefilter."""
    if cfg is None:
        cfg = RULE_CONFIG
    else:
        cfg = {**RULE_CONFIG, **cfg}

    df = df_windows.copy()
    df['rule_flag']   = None
    df['final_label'] = None

    def _flag(mask, rule_name, label_name):
        if mask is None or mask.sum() == 0:
            return
        m = mask & df['rule_flag'].isna()
        df.loc[m, 'rule_flag']   = rule_name
        df.loc[m, 'final_label'] = label_name

    # ============================================================
    # PRIORITY 1: VALUE RULES
    # ============================================================
    
    # 1.1 SATURATION_STUCK → SENSOR_FAILURE
    saturation_full = (
        (df['temp_max']  < cfg['saturation_max']) &
        (df['temp_min']  > cfg['saturation_min']) &
        (df['temp_mean'] < cfg['saturation_max'])
    )
    _flag(saturation_full, 'SATURATION_STUCK', 'SENSOR_FAILURE')

    # 1.2 MEAN_NEGATIVE_FLAKY → SENSOR_ANOMALY
    mean_negative_flaky = (df['temp_mean'] < 0) & (df['temp_max'] > 0)
    _flag(mean_negative_flaky, 'MEAN_NEGATIVE_FLAKY', 'SENSOR_ANOMALY')

    # 1.3 HAS_NEGATIVE → SENSOR_ANOMALY
    _flag(df['temp_min'] < 0, 'HAS_NEGATIVE', 'SENSOR_ANOMALY')

    # 1.4 HARD_OVERHEAT → SENSOR_ANOMALY
    _flag(df['temp_max'] > cfg['temp_overheat'], 'HARD_OVERHEAT', 'SENSOR_ANOMALY')

    # ============================================================
    # PRIORITY 2: GAP RULES
    # ============================================================
    
    has_required = all(
        c in df.columns for c in 
        ['num_data', 'n_valid_rh', 'n_valid_press',
         'max_gap_temp', 'max_gap_rh', 'max_gap_press']
    )
    
    if has_required:
        temp_low   = df['num_data']      < cfg['min_valid']
        rh_low     = df['n_valid_rh']    < cfg['min_valid']
        press_low  = df['n_valid_press'] < cfg['min_valid']
        
        temp_gapped   = df['max_gap_temp']  > cfg['major_gap_min']
        rh_gapped     = df['max_gap_rh']    > cfg['major_gap_min']
        press_gapped  = df['max_gap_press'] > cfg['major_gap_min']
        
        # 2.1 comm_few_rows_multisensor: SEMUA sensor low valid → GAP_FAULT
        few_rows_multi = temp_low & rh_low & press_low
        _flag(few_rows_multi, 'comm_few_rows_multisensor', 'GAP_FAULT')
        
        # 2.2 comm_major_gap_multisensor: SEMUA sensor major gap → GAP_FAULT
        major_gap_multi = temp_gapped & rh_gapped & press_gapped
        _flag(major_gap_multi, 'comm_major_gap_multisensor', 'GAP_FAULT')
        
        # 2.3 comm_few_rows_temp: TEMP only low valid → SENSOR_ANOMALY_GAP_DATA
        few_rows_temp = temp_low & ~rh_low & ~press_low
        _flag(few_rows_temp, 'comm_few_rows_temp', 'SENSOR_ANOMALY_GAP_DATA')
        
        # 2.4 comm_major_gap_temp: TEMP only gapped → SENSOR_ANOMALY_GAP_DATA
        major_gap_temp = temp_gapped & ~rh_gapped & ~press_gapped
        _flag(major_gap_temp, 'comm_major_gap_temp', 'SENSOR_ANOMALY_GAP_DATA')

    # ============================================================
    # PRIORITY 3: PROMOSI — value rule + gap → label gabungan
    # ============================================================
    
    if has_required:
        has_gap_issue = (
            (df['num_data']      < cfg['min_valid']) |
            (df['max_gap_temp']  > cfg['major_gap_min']) |
            (df['n_valid_rh']    < cfg['min_valid']) |
            (df['max_gap_rh']    > cfg['major_gap_min']) |
            (df['n_valid_press'] < cfg['min_valid']) |
            (df['max_gap_press'] > cfg['major_gap_min'])
        )
        
        sf_with_gap = (df['final_label'] == 'SENSOR_FAILURE') & has_gap_issue
        df.loc[sf_with_gap, 'final_label'] = 'SENSOR_FAILURE_WITH_GAP'
        
        sa_with_gap = (df['final_label'] == 'SENSOR_ANOMALY') & has_gap_issue
        df.loc[sa_with_gap, 'final_label'] = 'SENSOR_ANOMALY_WITH_GAP'

    df['is_rule_flagged'] = df['rule_flag'].notna()
    return df


def summarize_rules(df_windows, station_name=""):
    """Print breakdown rule_flag + final_label."""
    n_total = len(df_windows)
    flagged = df_windows[df_windows['is_rule_flagged']]
    n_flagged = len(flagged)
    print(f"  [RULE] {station_name}: {n_flagged}/{n_total} flagged "
          f"({100*n_flagged/n_total:.2f}%)")
    if n_flagged == 0:
        return
    print(f"         Per final_label:")
    for label, n in flagged['final_label'].value_counts().items():
        print(f"         - {label:30s}: {n:6d}")
    print(f"         Per rule_flag (granular):")
    for rule, n in flagged['rule_flag'].value_counts().items():
        print(f"         - {rule:30s}: {n:6d}")


# ============================================================
# PLOTTING HELPER
# ============================================================
def plot_rule_window(df_raw, idx, rule_flag, final_label, station_name, ax_temp, ax_rh, ax_sr):
    """Plot 1 window untuk verifikasi. Subplot temp + RH + SR."""
    window_start = idx - pd.Timedelta(WINDOW_SIZE)
    subset = df_raw.loc[window_start:idx]
    if len(subset) == 0:
        return False

    ax_temp.plot(subset.index, subset["temp"], lw=1.5, color="tab:blue", marker='o', markersize=2)
    ax_temp.axhline(0, color='red', lw=0.8, linestyle=':', alpha=0.5)
    ax_temp.set_ylabel("Temp (°C)")
    ax_temp.set_title(f"{rule_flag} ({final_label}) | {idx}", fontsize=10, fontweight="bold")
    ax_temp.grid(True, alpha=0.3)

    if "rh" in subset.columns:
        ax_rh.plot(subset.index, subset["rh"], lw=1.5, color="tab:green", marker='o', markersize=2)
    ax_rh.set_ylabel("RH (%)")
    ax_rh.grid(True, alpha=0.3)

    if "sr" in subset.columns:
        ax_sr.plot(subset.index, subset["sr"], lw=1.5, color="tab:orange", marker='o', markersize=2)
    ax_sr.set_ylabel("Solar (W/m²)")
    ax_sr.set_xlabel("Time (WIB)")
    ax_sr.grid(True, alpha=0.3)
    return True


def plot_rule_samples(df_raw, df_flagged, station_name, output_dir, samples_per_rule=20):
    """Plot max N sample per rule_flag ke PDF."""
    if len(df_flagged) == 0:
        print(f"  [PLOT] Tidak ada window flagged, skip PDF.")
        return

    pdf_path = os.path.join(
        output_dir,
        f"rule_verification_{station_name.replace(' ', '_')}.pdf"
    )
    print(f"  [PLOT] Generating {pdf_path}...")

    with PdfPages(pdf_path) as pdf:
        for rule in sorted(df_flagged['rule_flag'].dropna().unique()):
            df_rule = df_flagged[df_flagged['rule_flag'] == rule]
            n_samples = min(samples_per_rule, len(df_rule))
            
            if len(df_rule) > samples_per_rule:
                idx_samples = df_rule.sample(n=samples_per_rule, random_state=42).index.sort_values()
            else:
                idx_samples = df_rule.index
            
            print(f"     - {rule}: plotting {n_samples}/{len(df_rule)} samples")
            
            for idx in idx_samples:
                fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(11, 8), sharex=True)
                final_label = df_rule.loc[idx, 'final_label']
                ok = plot_rule_window(df_raw, idx, rule, final_label, station_name, ax1, ax2, ax3)
                if ok:
                    fig.suptitle(f"{station_name} — Rule Verification", fontsize=11, fontweight='bold')
                    plt.tight_layout()
                    pdf.savefig(fig, bbox_inches="tight")
                plt.close(fig)
    print(f"  [PLOT] Done.")

---
## STAGE 1: RULE-BASED PIPELINE

In [7]:
# ============================================================
# CELL 7: MAIN LOOP — RULE-BASED
# ============================================================

all_windows = {}   # {station: df_windows (full, with rule_flag)}
all_raw     = {}   # {station: df_raw}

for station_name, file_path in station_files.items():
    print(f"\n{'='*60}")
    print(f"RULE-BASED: {station_name.upper()}")
    print(f"{'='*60}")

    try:
        # A. Preprocess + window dataset
        df_raw = preprocess(file_path)
        df_windows = build_window_dataset(df_raw, station_name)
        if len(df_windows) == 0:
            print(f"  [SKIP] Tidak ada window valid.")
            continue

        # B. Apply rule prefilter
        df_windows = apply_rule_prefilter(df_windows)
        summarize_rules(df_windows, station_name)

        # Simpan ke accumulator
        all_windows[station_name] = df_windows
        all_raw[station_name]     = df_raw

        # C. Export Excel
        df_flagged = df_windows[df_windows['is_rule_flagged']].copy()
        if len(df_flagged) == 0:
            print(f"  [EXCEL] Tidak ada window flagged, skip.")
            continue

        # Rule descriptions
        rule_descriptions = {
            # Value rules
            'SATURATION_STUCK':           f"temp_min, temp_max, temp_mean SEMUA di range {RULE_CONFIG['saturation_min']}..{RULE_CONFIG['saturation_max']} → sensor stuck saturasi",
            'MEAN_NEGATIVE_FLAKY':        'temp_mean < 0 TAPI temp_max > 0 → sensor flaky (mostly negative tapi sesekali valid)',
            'HAS_NEGATIVE':               'temp_min < 0 dengan mean positive (transient: 1-2 spike negative)',
            'HARD_OVERHEAT':              f"temp_max > {RULE_CONFIG['temp_overheat']}°C → shield/exposure issue",
            # Gap rules
            'comm_few_rows_multisensor':  f"SEMUA sensor (temp, rh, press) valid count < {RULE_CONFIG['min_valid']} → datalogger mostly down",
            'comm_major_gap_multisensor': f"SEMUA sensor max_gap > {RULE_CONFIG['major_gap_min']} menit → datalogger outage",
            'comm_few_rows_temp':         f"TEMP valid count < {RULE_CONFIG['min_valid']}, RH/press normal → temp sensor specifically",
            'comm_major_gap_temp':        f"TEMP max_gap > {RULE_CONFIG['major_gap_min']} menit, RH/press normal → temp sensor specifically",
        }
        df_flagged['rule_description'] = df_flagged['rule_flag'].map(rule_descriptions)

        # Output columns (urutan untuk Excel)
        output_cols = [
            'nama_site',
            'WIB_start_time',
            'WIB_end_time',
            'window_start_utc',
            'window_end_utc',
            'final_label',
            'rule_flag',
            'rule_description',
            # Fitur dasar
            'temp_mean', 'temp_std', 'temp_min', 'temp_max', 'temp_range',
            # Fitur gap (per sensor)
            'n_rows', 'sampling_rate',
            'num_data', 'n_valid_rh', 'n_valid_press', 'n_valid_sr',
            'max_gap_temp', 'max_gap_rh', 'max_gap_press',
            'n_gaps', 'max_gap_min',
            # Sensor at_end (referensi)
            'temp_at_end', 'rh_at_end', 'sr_at_end',
        ]
        output_cols = [c for c in output_cols if c in df_flagged.columns]
        df_export   = df_flagged[output_cols].copy()

        fname = os.path.join(
            RULE_DIR,
            f"rule_based_{station_name.replace(' ', '_')}.xlsx"
        )
        with pd.ExcelWriter(fname, engine="openpyxl") as writer:
            df_export.to_excel(writer, sheet_name="ALL", index=False)
            for label in df_export['final_label'].dropna().unique():
                df_label = df_export[df_export['final_label'] == label]
                sheet = label[:31]
                df_label.to_excel(writer, sheet_name=sheet, index=False)
        print(f"  [EXCEL] {len(df_export)} windows → {fname}")

        # D. Plot 20 sample per rule_flag
        plot_rule_samples(df_raw, df_flagged, station_name, RULE_DIR, samples_per_rule=20)

    except Exception as e:
        import traceback
        print(f"  [ERROR] {station_name}: {e}")
        traceback.print_exc()

print(f"\n{'='*60}")
print(f"STAGE 1 SELESAI. Output: {RULE_DIR}")
print(f"{'='*60}")


RULE-BASED: CIBADAK
  [INFO] Gap > 1 menit: 1905
  [INFO] Duplikat ditemukan: 0
  [INFO] Total windows: 3535
  [RULE] Cibadak: 122/3535 flagged (3.45%)
         Per final_label:
         - GAP_FAULT                     :    122
         Per rule_flag (granular):
         - comm_few_rows_multisensor     :    122
  [EXCEL] 122 windows → output_pipeline_v3\01_rule_based_\rule_based_Cibadak.xlsx
  [PLOT] Generating output_pipeline_v3\01_rule_based_\rule_verification_Cibadak.pdf...
     - comm_few_rows_multisensor: plotting 20/122 samples
  [PLOT] Done.

RULE-BASED: CIBALIUNG
  [INFO] Gap > 1 menit: 15886
  [INFO] Duplikat ditemukan: 18
         - Lossless (nilai identik): 18
         - Konflik (nilai berbeda):  0
  [INFO] Sisa duplikat setelah keep-first: 0
  [INFO] Total windows: 20428
  [RULE] Cibaliung: 1518/20428 flagged (7.43%)
         Per final_label:
         - GAP_FAULT                     :    788
         - SENSOR_FAILURE                :    574
         - SENSOR_ANOMALY       

---
## STAGE 2: ML PIPELINE (data clean)

In [8]:
# ============================================================
# CELL 9: MAIN LOOP — ML ENSEMBLE (PADA DATA BERSIH)
# ============================================================

all_ml_results = {}   # {station: df_ml_result}

for station_name in station_files.keys():
    print(f"\n{'='*60}")
    print(f"ML: {station_name.upper()}")
    print(f"{'='*60}")

    if station_name not in all_windows:
        print(f"  [SKIP] Stasiun belum diproses di Stage 1.")
        continue

    try:
        df_windows = all_windows[station_name]
        df_raw     = all_raw[station_name]

        # A. Filter: hanya window yang LOLOS rule (data bersih)
        df_clean = df_windows[~df_windows['is_rule_flagged']].copy()
        n_clean   = len(df_clean)
        n_flagged = len(df_windows) - n_clean
        print(f"  [DATA] Total: {len(df_windows)} | Flagged (skip): {n_flagged} | Clean → ML: {n_clean}")

        if n_clean < 100:
            print(f"  [SKIP] Data clean terlalu sedikit untuk ML.")
            continue

        # B. Feature matrix
        feature_cols = [c for c in df_clean.columns if c not in EXCLUDE_COLS]
        X = df_clean[feature_cols].dropna().copy()
        if len(X) == 0:
            print(f"  [SKIP] Data kosong setelah dropna.")
            continue

        non_numeric = X.select_dtypes(exclude=[np.number]).columns.tolist()
        if non_numeric:
            print(f"  [WARN] Drop non-numeric cols: {non_numeric}")
            X = X.drop(columns=non_numeric)

        print(f"  [ML] Train shape: {X.shape}")
        print(f"  [ML] Features: {list(X.columns)}")

        # C. Scaling
        scaler = StandardScaler()
        X_scaled = pd.DataFrame(scaler.fit_transform(X), index=X.index, columns=X.columns)

        # D. Isolation Forest
        iso = IsolationForest(n_estimators=200, contamination=0.001, random_state=42)
        iso.fit(X_scaled)
        anomaly_if = pd.Series((iso.predict(X_scaled) == -1).astype(int), index=X.index)
        score_if   = pd.Series(iso.decision_function(X_scaled), index=X.index)

        # E. LOF
        lof = LocalOutlierFactor(n_neighbors=20, contamination=0.001, n_jobs=-1)
        anomaly_lof = pd.Series((lof.fit_predict(X_scaled) == -1).astype(int), index=X.index)
        score_lof   = pd.Series(lof.negative_outlier_factor_, index=X.index)

        # F. OC-SVM
        ocsvm = OneClassSVM(kernel="rbf", gamma="scale", nu=0.001)
        if len(X_scaled) > 100_000:
            ocsvm.fit(X_scaled.sample(n=100_000, random_state=42))
        else:
            ocsvm.fit(X_scaled)
        anomaly_ocsvm = pd.Series((ocsvm.predict(X_scaled) == -1).astype(int), index=X.index)
        score_ocsvm   = pd.Series(ocsvm.decision_function(X_scaled), index=X.index)

        # G. Voting + Label
        votes  = anomaly_if + anomaly_lof + anomaly_ocsvm
        label_map = {3: "STRONG", 2: "MEDIUM", 1: "WEAK", 0: "NORMAL"}
        labels = votes.map(label_map)

        print(f"  STRONG  : {(votes==3).sum()}")
        print(f"  MEDIUM  : {(votes==2).sum()}")
        print(f"  WEAK    : {(votes==1).sum()}")
        print(f"  NORMAL  : {(votes==0).sum()}")

        # H. Build result df
        df_result = pd.DataFrame({
            "anomaly_if"   : anomaly_if,
            "score_if"     : score_if,
            "anomaly_lof"  : anomaly_lof,
            "score_lof"    : score_lof,
            "anomaly_ocsvm": anomaly_ocsvm,
            "score_ocsvm"  : score_ocsvm,
            "anomaly_votes": votes,
            "anomaly_label": labels,
        }, index=X.index)
        all_ml_results[station_name] = df_result

        # I. Export Excel — SEMUA window dengan label ML
        df_export = df_clean.join(df_result, how='inner')
        
        if len(df_export) == 0:
            print(f"  [EXCEL] Empty join, skip Excel.")
        else:
            ID_COLS = [
                'nama_site',
                'WIB_start_time',
                'WIB_end_time',
                'window_start_utc',
                'window_end_utc',
            ]
            META_COLS = [
                'hour_index',
                'num_data',
                'sampling_rate',
            ]
            ML_COLS = [
                'anomaly_label', 'anomaly_votes',
                'anomaly_if', 'score_if',
                'anomaly_lof', 'score_lof',
                'anomaly_ocsvm', 'score_ocsvm',
            ]
            FEATURE_COLS = list(X.columns)  # semua fitur ML
            CONTEXT_COLS = [
                'rr_sum', 'rr_max_rate',
                'sr_mean', 'sr_max', 'sr_std',
                'rh_mean', 'rh_min', 'rh_max', 'rh_std',
            ]
            
            output_cols_raw = ID_COLS + META_COLS + ML_COLS + FEATURE_COLS + CONTEXT_COLS
            seen = set()
            output_cols = [c for c in output_cols_raw 
                           if (c in df_export.columns and c not in seen and not seen.add(c))]
            df_export_filtered = df_export[output_cols]

            fname = os.path.join(
                ML_DIR,
                f"ml_results_{station_name.replace(' ', '_')}.xlsx"
            )
            with pd.ExcelWriter(fname, engine="openpyxl") as writer:
                df_export_filtered.to_excel(writer, sheet_name="ALL", index=False)
                for lbl in ['STRONG', 'MEDIUM', 'WEAK', 'NORMAL']:
                    df_lbl = df_export_filtered[df_export_filtered['anomaly_label'] == lbl]
                    if len(df_lbl) > 0:
                        df_lbl.to_excel(writer, sheet_name=lbl, index=False)
            
            n_strong = (df_export_filtered['anomaly_label'] == 'STRONG').sum()
            n_medium = (df_export_filtered['anomaly_label'] == 'MEDIUM').sum()
            n_weak   = (df_export_filtered['anomaly_label'] == 'WEAK').sum()
            n_normal = (df_export_filtered['anomaly_label'] == 'NORMAL').sum()
            print(f"  [EXCEL] {len(df_export_filtered)} windows "
                  f"(S={n_strong} M={n_medium} W={n_weak} N={n_normal}) → {fname}")

    except Exception as e:
        import traceback
        print(f"  [ERROR] {station_name}: {e}")
        traceback.print_exc()

print(f"\n{'='*60}")
print(f"STAGE 2 ML SELESAI. Output: {ML_DIR}")
print(f"{'='*60}")


ML: CIBADAK
  [DATA] Total: 3535 | Flagged (skip): 122 | Clean → ML: 3413
  [ML] Train shape: (3413, 19)
  [ML] Features: ['temp_mean', 'temp_std', 'temp_min', 'temp_max', 'temp_range', 'temp_skew', 'temp_kurt', 'temp_cv', 'delta_T_mean', 'delta_T_std', 'delta_T_abs_max', 'delta_T_per_minute_max', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'consecutive_identical', '1h_unique_ratio', 'temp_zscore_max']
  STRONG  : 0
  MEDIUM  : 5
  WEAK    : 62
  NORMAL  : 3346
  [EXCEL] 3413 windows (S=0 M=5 W=62 N=3346) → output_pipeline_v3\02_ml_\ml_results_Cibadak.xlsx

ML: CIBALIUNG
  [DATA] Total: 20428 | Flagged (skip): 1518 | Clean → ML: 18910
  [ML] Train shape: (18910, 19)
  [ML] Features: ['temp_mean', 'temp_std', 'temp_min', 'temp_max', 'temp_range', 'temp_skew', 'temp_kurt', 'temp_cv', 'delta_T_mean', 'delta_T_std', 'delta_T_abs_max', 'delta_T_per_minute_max', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'consecutive_identical', '1h_unique_ratio', 'temp_zscore_max']
  STRONG  :

In [9]:
# ============================================================
# CELL 11: COMBINED PLOTTING — STRONG, WEAK, MEDIUM, NORMAL
# ============================================================
# Semua kategori jadi 1 PDF combined per kategori (semua stasiun):
#   STRONG: semua sample
#   MEDIUM: semua sample
#   WEAK  : semua sample
#   NORMAL: 5 sample per stasiun (sanity check)
# ============================================================

# === 1. STRONG — combined PDF, SEMUA sample ===
print("\n--- STRONG plot (combined) ---")
strong_pdf_path = os.path.join(ML_DIR, "ml_strong_ALL_STATIONS.pdf")
n_strong_total = 0
with PdfPages(strong_pdf_path) as pdf:
    for station_name, df_result in all_ml_results.items():
        if station_name not in all_raw:
            continue
        df_raw = all_raw[station_name]
        df_strong = df_result[df_result['anomaly_votes'] == 3]
        if len(df_strong) == 0:
            continue
        
        idx_samples = df_strong.index.sort_values()
        print(f"  {station_name}: plotting {len(idx_samples)} STRONG samples...")
        for idx in idx_samples:
            fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(11, 8), sharex=True)
            plot_rule_window(df_raw, idx, 'ML_STRONG', 'STRONG', station_name, ax1, ax2, ax3)
            fig.suptitle(f"{station_name} — ML STRONG (3/3 votes)", fontsize=11, fontweight='bold')
            plt.tight_layout()
            pdf.savefig(fig, bbox_inches="tight")
            plt.close(fig)
            n_strong_total += 1
print(f"[PDF] Total {n_strong_total} STRONG samples → {strong_pdf_path}")

# === 2. MEDIUM — combined PDF, SEMUA sample ===
print("\n--- MEDIUM plot (combined) ---")
medium_pdf_path = os.path.join(ML_DIR, "ml_medium_ALL_STATIONS.pdf")
n_medium_total = 0
with PdfPages(medium_pdf_path) as pdf:
    for station_name, df_result in all_ml_results.items():
        if station_name not in all_raw:
            continue
        df_raw = all_raw[station_name]
        df_medium = df_result[df_result['anomaly_votes'] == 2]
        if len(df_medium) == 0:
            continue
        
        idx_samples = df_medium.index.sort_values()
        print(f"  {station_name}: plotting {len(idx_samples)} MEDIUM samples...")
        for idx in idx_samples:
            fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(11, 8), sharex=True)
            plot_rule_window(df_raw, idx, 'ML_MEDIUM', 'MEDIUM', station_name, ax1, ax2, ax3)
            fig.suptitle(f"{station_name} — ML MEDIUM (2/3 votes)", fontsize=11, fontweight='bold')
            plt.tight_layout()
            pdf.savefig(fig, bbox_inches="tight")
            plt.close(fig)
            n_medium_total += 1
print(f"[PDF] Total {n_medium_total} MEDIUM samples → {medium_pdf_path}")

# === 3. WEAK — combined PDF, SEMUA sample ===
print("\n--- WEAK plot (combined) ---")
weak_pdf_path = os.path.join(ML_DIR, "ml_weak_ALL_STATIONS.pdf")
n_weak_total = 0
with PdfPages(weak_pdf_path) as pdf:
    for station_name, df_result in all_ml_results.items():
        if station_name not in all_raw:
            continue
        df_raw = all_raw[station_name]
        df_weak = df_result[df_result['anomaly_votes'] == 1]
        if len(df_weak) == 0:
            continue
        
        idx_samples = df_weak.index.sort_values()
        print(f"  {station_name}: plotting {len(idx_samples)} WEAK samples...")
        for idx in idx_samples:
            fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(11, 8), sharex=True)
            plot_rule_window(df_raw, idx, 'ML_WEAK', 'WEAK', station_name, ax1, ax2, ax3)
            fig.suptitle(f"{station_name} — ML WEAK (1/3 votes)", fontsize=11, fontweight='bold')
            plt.tight_layout()
            pdf.savefig(fig, bbox_inches="tight")
            plt.close(fig)
            n_weak_total += 1
print(f"[PDF] Total {n_weak_total} WEAK samples → {weak_pdf_path}")

# === 4. NORMAL — combined PDF, 5 sample per stasiun ===
print("\n--- NORMAL plot (combined, 5 sample/stasiun) ---")
normal_pdf_path = os.path.join(ML_DIR, "ml_normal_ALL_STATIONS.pdf")
n_normal_total = 0
with PdfPages(normal_pdf_path) as pdf:
    for station_name, df_result in all_ml_results.items():
        if station_name not in all_raw:
            continue
        df_raw = all_raw[station_name]
        df_normal = df_result[df_result['anomaly_votes'] == 0]
        if len(df_normal) == 0:
            continue
        
        n_plot = min(5, len(df_normal))
        idx_samples = df_normal.sample(n=n_plot, random_state=42).index.sort_values()
        for idx in idx_samples:
            fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(11, 8), sharex=True)
            plot_rule_window(df_raw, idx, 'ML_NORMAL', 'NORMAL', station_name, ax1, ax2, ax3)
            fig.suptitle(f"{station_name} — ML NORMAL (0/3 votes)", fontsize=11, fontweight='bold')
            plt.tight_layout()
            pdf.savefig(fig, bbox_inches="tight")
            plt.close(fig)
            n_normal_total += 1
print(f"[PDF] Total {n_normal_total} NORMAL samples → {normal_pdf_path}")

print(f"\n{'='*60}")
print(f"PLOTTING SELESAI. Output: {ML_DIR}")
print(f"{'='*60}")


--- STRONG plot (combined) ---
  Cibaliung: plotting 3 STRONG samples...
  Cileles: plotting 2 STRONG samples...
  Leuwi Dimar: plotting 1 STRONG samples...
  Singamerta: plotting 2 STRONG samples...
  BSD Serpong: plotting 1 STRONG samples...
  Golf Modern: plotting 3 STRONG samples...
  PIK: plotting 1 STRONG samples...
  Staklim Banten: plotting 2 STRONG samples...
[PDF] Total 15 STRONG samples → output_pipeline_v3\02_ml_\ml_strong_ALL_STATIONS.pdf

--- MEDIUM plot (combined) ---
  Cibadak: plotting 5 MEDIUM samples...
  Cibaliung: plotting 14 MEDIUM samples...
  Cileles: plotting 14 MEDIUM samples...
  Leuwi Dimar: plotting 17 MEDIUM samples...
  Singamerta: plotting 13 MEDIUM samples...
  Kebun Bibit Cibubur: plotting 15 MEDIUM samples...
  BSD Serpong: plotting 25 MEDIUM samples...
  Golf Modern: plotting 17 MEDIUM samples...
  PIK: plotting 10 MEDIUM samples...
  TMII: plotting 10 MEDIUM samples...
  Staklim Banten: plotting 16 MEDIUM samples...
[PDF] Total 156 MEDIUM samples →

In [10]:
# ============================================================
# CELL 13: COMBINED SUMMARY
# ============================================================

summary_rows = []

for station_name in station_files.keys():
    if station_name not in all_windows:
        continue
    
    df_w = all_windows[station_name]
    n_total   = len(df_w)
    n_flagged = df_w['is_rule_flagged'].sum()
    
    # Per-label rule counts
    label_counts = df_w[df_w['is_rule_flagged']]['final_label'].value_counts().to_dict()
    
    # ML counts
    if station_name in all_ml_results:
        df_ml = all_ml_results[station_name]
        ml_strong = (df_ml['anomaly_votes'] == 3).sum()
        ml_medium = (df_ml['anomaly_votes'] == 2).sum()
        ml_weak   = (df_ml['anomaly_votes'] == 1).sum()
        ml_normal = (df_ml['anomaly_votes'] == 0).sum()
    else:
        ml_strong = ml_medium = ml_weak = ml_normal = 0
    
    summary_rows.append({
        'station': station_name,
        'total_windows': n_total,
        'rule_flagged': n_flagged,
        'rule_pct':     round(100 * n_flagged / n_total, 2),
        'SENSOR_FAILURE':              label_counts.get('SENSOR_FAILURE', 0),
        'SENSOR_FAILURE_WITH_GAP':     label_counts.get('SENSOR_FAILURE_WITH_GAP', 0),
        'SENSOR_ANOMALY':              label_counts.get('SENSOR_ANOMALY', 0),
        'SENSOR_ANOMALY_WITH_GAP':     label_counts.get('SENSOR_ANOMALY_WITH_GAP', 0),
        'GAP_FAULT':                   label_counts.get('GAP_FAULT', 0),
        'SENSOR_ANOMALY_GAP_DATA':     label_counts.get('SENSOR_ANOMALY_GAP_DATA', 0),
        'ml_input':    n_total - n_flagged,
        'ml_normal':   ml_normal,
        'ml_weak':     ml_weak,
        'ml_medium':   ml_medium,
        'ml_strong':   ml_strong,
    })

df_summary = pd.DataFrame(summary_rows)
print("\n=== SUMMARY ALL STATIONS ===")
print(df_summary.to_string(index=False))

summary_path = os.path.join(OUTPUT_DIR, "summary_all_stations.xlsx")
df_summary.to_excel(summary_path, index=False)
print(f"\n[EXCEL] Summary → {summary_path}")


=== SUMMARY ALL STATIONS ===
            station  total_windows  rule_flagged  rule_pct  SENSOR_FAILURE  SENSOR_FAILURE_WITH_GAP  SENSOR_ANOMALY  SENSOR_ANOMALY_WITH_GAP  GAP_FAULT  SENSOR_ANOMALY_GAP_DATA  ml_input  ml_normal  ml_weak  ml_medium  ml_strong
            Cibadak           3535           122      3.45               0                        0               0                        0        122                        0      3413       3346       62          5          0
          Cibaliung          20428          1518      7.43             574                       11             139                        6        788                        0     18910      18787      106         14          3
            Cileles          22607          8004     35.40            3927                      136             318                       18       3605                        0     14603      14489       98         14          2
        Leuwi Dimar          22823          1232      

In [11]:
# ============================================================
# CELL TAMBAHAN: RE-RUN ML PIPELINE 1 dengan contamination=0.01
# ============================================================
# Pipeline 1 sebelumnya pakai contamination=0.001 (konservatif).
# Cell ini re-run ML pakai contamination=0.01 (lebih longgar) untuk
# konsistensi dengan eksperimen Pipeline 2.
#
# Output disimpan di folder terpisah (02_ml_contam_0.01) biar tidak
# overwrite hasil contamination=0.001 sebelumnya.
#
# Pakai data dari memory (all_windows, all_raw) — TIDAK perlu re-run
# preprocessing atau rule-based.
# ============================================================

CONTAMINATION_NEW = 0.01

# Output dir terpisah
ML_DIR_V2 = os.path.join(OUTPUT_DIR, f"02_ml_contam_{CONTAMINATION_NEW}")
os.makedirs(ML_DIR_V2, exist_ok=True)

all_ml_results_v2 = {}

for station_name in station_files.keys():
    print(f"\n{'='*60}")
    print(f"ML (contam={CONTAMINATION_NEW}): {station_name.upper()}")
    print(f"{'='*60}")

    if station_name not in all_windows:
        print(f"  [SKIP] Stasiun belum diproses di Stage 1.")
        continue

    try:
        df_windows = all_windows[station_name]
        df_raw     = all_raw[station_name]

        df_clean = df_windows[~df_windows['is_rule_flagged']].copy()
        n_clean   = len(df_clean)
        n_flagged = len(df_windows) - n_clean
        print(f"  [DATA] Total: {len(df_windows)} | Flagged (skip): {n_flagged} | Clean → ML: {n_clean}")

        if n_clean < 100:
            print(f"  [SKIP] Data clean terlalu sedikit untuk ML.")
            continue

        feature_cols = [c for c in df_clean.columns if c not in EXCLUDE_COLS]
        X = df_clean[feature_cols].dropna().copy()
        if len(X) == 0:
            print(f"  [SKIP] Data kosong setelah dropna.")
            continue

        non_numeric = X.select_dtypes(exclude=[np.number]).columns.tolist()
        if non_numeric:
            X = X.drop(columns=non_numeric)

        print(f"  [ML] Train shape: {X.shape}")

        scaler = StandardScaler()
        X_scaled = pd.DataFrame(scaler.fit_transform(X), index=X.index, columns=X.columns)

        # Pakai contamination baru
        iso = IsolationForest(n_estimators=200, contamination=CONTAMINATION_NEW, random_state=42)
        iso.fit(X_scaled)
        anomaly_if = pd.Series((iso.predict(X_scaled) == -1).astype(int), index=X.index)
        score_if   = pd.Series(iso.decision_function(X_scaled), index=X.index)

        lof = LocalOutlierFactor(n_neighbors=20, contamination=CONTAMINATION_NEW, n_jobs=-1)
        anomaly_lof = pd.Series((lof.fit_predict(X_scaled) == -1).astype(int), index=X.index)
        score_lof   = pd.Series(lof.negative_outlier_factor_, index=X.index)

        ocsvm = OneClassSVM(kernel="rbf", gamma="scale", nu=CONTAMINATION_NEW)
        if len(X_scaled) > 100_000:
            ocsvm.fit(X_scaled.sample(n=100_000, random_state=42))
        else:
            ocsvm.fit(X_scaled)
        anomaly_ocsvm = pd.Series((ocsvm.predict(X_scaled) == -1).astype(int), index=X.index)
        score_ocsvm   = pd.Series(ocsvm.decision_function(X_scaled), index=X.index)

        votes  = anomaly_if + anomaly_lof + anomaly_ocsvm
        label_map = {3: "STRONG", 2: "MEDIUM", 1: "WEAK", 0: "NORMAL"}
        labels = votes.map(label_map)

        print(f"  STRONG  : {(votes==3).sum()}")
        print(f"  MEDIUM  : {(votes==2).sum()}")
        print(f"  WEAK    : {(votes==1).sum()}")
        print(f"  NORMAL  : {(votes==0).sum()}")

        df_result = pd.DataFrame({
            "anomaly_if"   : anomaly_if,
            "score_if"     : score_if,
            "anomaly_lof"  : anomaly_lof,
            "score_lof"    : score_lof,
            "anomaly_ocsvm": anomaly_ocsvm,
            "score_ocsvm"  : score_ocsvm,
            "anomaly_votes": votes,
            "anomaly_label": labels,
        }, index=X.index)
        all_ml_results_v2[station_name] = df_result

        # Export Excel
        df_export = df_clean.join(df_result, how='inner')
        if len(df_export) > 0:
            ID_COLS = ['nama_site', 'WIB_start_time', 'WIB_end_time',
                       'window_start_utc', 'window_end_utc']
            META_COLS = ['hour_index', 'num_data', 'sampling_rate']
            ML_COLS = ['anomaly_label', 'anomaly_votes',
                       'anomaly_if', 'score_if',
                       'anomaly_lof', 'score_lof',
                       'anomaly_ocsvm', 'score_ocsvm']
            FEATURE_COLS = list(X.columns)
            CONTEXT_COLS = ['rr_sum', 'rr_max_rate',
                            'sr_mean', 'sr_max', 'sr_std',
                            'rh_mean', 'rh_min', 'rh_max', 'rh_std']
            
            output_cols_raw = ID_COLS + META_COLS + ML_COLS + FEATURE_COLS + CONTEXT_COLS
            seen = set()
            output_cols = [c for c in output_cols_raw 
                           if (c in df_export.columns and c not in seen and not seen.add(c))]
            df_export_filtered = df_export[output_cols]

            fname = os.path.join(
                ML_DIR_V2,
                f"ml_results_{station_name.replace(' ', '_')}.xlsx"
            )
            with pd.ExcelWriter(fname, engine="openpyxl") as writer:
                df_export_filtered.to_excel(writer, sheet_name="ALL", index=False)
                for lbl in ['STRONG', 'MEDIUM', 'WEAK', 'NORMAL']:
                    df_lbl = df_export_filtered[df_export_filtered['anomaly_label'] == lbl]
                    if len(df_lbl) > 0:
                        df_lbl.to_excel(writer, sheet_name=lbl, index=False)
            
            n_strong = (df_export_filtered['anomaly_label'] == 'STRONG').sum()
            n_medium = (df_export_filtered['anomaly_label'] == 'MEDIUM').sum()
            n_weak   = (df_export_filtered['anomaly_label'] == 'WEAK').sum()
            n_normal = (df_export_filtered['anomaly_label'] == 'NORMAL').sum()
            print(f"  [EXCEL] {len(df_export_filtered)} windows "
                  f"(S={n_strong} M={n_medium} W={n_weak} N={n_normal}) → {fname}")

    except Exception as e:
        import traceback
        print(f"  [ERROR] {station_name}: {e}")
        traceback.print_exc()

print(f"\n{'='*60}")
print(f"RE-RUN ML SELESAI. Output: {ML_DIR_V2}")
print(f"{'='*60}")


ML (contam=0.01): CIBADAK
  [DATA] Total: 3535 | Flagged (skip): 122 | Clean → ML: 3413
  [ML] Train shape: (3413, 19)
  STRONG  : 4
  MEDIUM  : 22
  WEAK    : 83
  NORMAL  : 3304
  [EXCEL] 3413 windows (S=4 M=22 W=83 N=3304) → output_pipeline_v3\02_ml_contam_0.01\ml_results_Cibadak.xlsx

ML (contam=0.01): CIBALIUNG
  [DATA] Total: 20428 | Flagged (skip): 1518 | Clean → ML: 18910
  [ML] Train shape: (18910, 19)
  STRONG  : 28
  MEDIUM  : 93
  WEAK    : 309
  NORMAL  : 18480
  [EXCEL] 18910 windows (S=28 M=93 W=309 N=18480) → output_pipeline_v3\02_ml_contam_0.01\ml_results_Cibaliung.xlsx

ML (contam=0.01): CILELES
  [DATA] Total: 22607 | Flagged (skip): 8004 | Clean → ML: 14603
  [ML] Train shape: (14603, 19)
  STRONG  : 27
  MEDIUM  : 73
  WEAK    : 221
  NORMAL  : 14282
  [EXCEL] 14603 windows (S=27 M=73 W=221 N=14282) → output_pipeline_v3\02_ml_contam_0.01\ml_results_Cileles.xlsx

ML (contam=0.01): LEUWI DIMAR
  [DATA] Total: 22823 | Flagged (skip): 1232 | Clean → ML: 21591
  [ML] T

In [12]:
# ============================================================
# CELL TAMBAHAN: PLOTTING untuk hasil contamination=0.01
# ============================================================

# 1. STRONG
print("\n--- STRONG plot (combined) ---")
strong_pdf_path = os.path.join(ML_DIR_V2, "ml_strong_ALL_STATIONS.pdf")
n_strong_total = 0
with PdfPages(strong_pdf_path) as pdf:
    for station_name, df_result in all_ml_results_v2.items():
        if station_name not in all_raw:
            continue
        df_raw = all_raw[station_name]
        df_strong = df_result[df_result['anomaly_votes'] == 3]
        if len(df_strong) == 0:
            continue
        idx_samples = df_strong.index.sort_values()
        print(f"  {station_name}: plotting {len(idx_samples)} STRONG samples...")
        for idx in idx_samples:
            fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(11, 8), sharex=True)
            plot_rule_window(df_raw, idx, 'ML_STRONG', 'STRONG', station_name, ax1, ax2, ax3)
            fig.suptitle(f"{station_name} — ML STRONG (3/3 votes)", fontsize=11, fontweight='bold')
            plt.tight_layout()
            pdf.savefig(fig, bbox_inches="tight")
            plt.close(fig)
            n_strong_total += 1
print(f"[PDF] Total {n_strong_total} STRONG samples → {strong_pdf_path}")

# 2. MEDIUM
print("\n--- MEDIUM plot (combined) ---")
medium_pdf_path = os.path.join(ML_DIR_V2, "ml_medium_ALL_STATIONS.pdf")
n_medium_total = 0
with PdfPages(medium_pdf_path) as pdf:
    for station_name, df_result in all_ml_results_v2.items():
        if station_name not in all_raw:
            continue
        df_raw = all_raw[station_name]
        df_medium = df_result[df_result['anomaly_votes'] == 2]
        if len(df_medium) == 0:
            continue
        idx_samples = df_medium.index.sort_values()
        print(f"  {station_name}: plotting {len(idx_samples)} MEDIUM samples...")
        for idx in idx_samples:
            fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(11, 8), sharex=True)
            plot_rule_window(df_raw, idx, 'ML_MEDIUM', 'MEDIUM', station_name, ax1, ax2, ax3)
            fig.suptitle(f"{station_name} — ML MEDIUM (2/3 votes)", fontsize=11, fontweight='bold')
            plt.tight_layout()
            pdf.savefig(fig, bbox_inches="tight")
            plt.close(fig)
            n_medium_total += 1
print(f"[PDF] Total {n_medium_total} MEDIUM samples → {medium_pdf_path}")

# 3. WEAK
print("\n--- WEAK plot (combined) ---")
weak_pdf_path = os.path.join(ML_DIR_V2, "ml_weak_ALL_STATIONS.pdf")
n_weak_total = 0
with PdfPages(weak_pdf_path) as pdf:
    for station_name, df_result in all_ml_results_v2.items():
        if station_name not in all_raw:
            continue
        df_raw = all_raw[station_name]
        df_weak = df_result[df_result['anomaly_votes'] == 1]
        if len(df_weak) == 0:
            continue
        idx_samples = df_weak.index.sort_values()
        print(f"  {station_name}: plotting {len(idx_samples)} WEAK samples...")
        for idx in idx_samples:
            fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(11, 8), sharex=True)
            plot_rule_window(df_raw, idx, 'ML_WEAK', 'WEAK', station_name, ax1, ax2, ax3)
            fig.suptitle(f"{station_name} — ML WEAK (1/3 votes)", fontsize=11, fontweight='bold')
            plt.tight_layout()
            pdf.savefig(fig, bbox_inches="tight")
            plt.close(fig)
            n_weak_total += 1
print(f"[PDF] Total {n_weak_total} WEAK samples → {weak_pdf_path}")

# 4. NORMAL — 5 sample per stasiun
print("\n--- NORMAL plot (combined, 5 sample/stasiun) ---")
normal_pdf_path = os.path.join(ML_DIR_V2, "ml_normal_ALL_STATIONS.pdf")
n_normal_total = 0
with PdfPages(normal_pdf_path) as pdf:
    for station_name, df_result in all_ml_results_v2.items():
        if station_name not in all_raw:
            continue
        df_raw = all_raw[station_name]
        df_normal = df_result[df_result['anomaly_votes'] == 0]
        if len(df_normal) == 0:
            continue
        n_plot = min(5, len(df_normal))
        idx_samples = df_normal.sample(n=n_plot, random_state=42).index.sort_values()
        for idx in idx_samples:
            fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(11, 8), sharex=True)
            plot_rule_window(df_raw, idx, 'ML_NORMAL', 'NORMAL', station_name, ax1, ax2, ax3)
            fig.suptitle(f"{station_name} — ML NORMAL (0/3 votes)", fontsize=11, fontweight='bold')
            plt.tight_layout()
            pdf.savefig(fig, bbox_inches="tight")
            plt.close(fig)
            n_normal_total += 1
print(f"[PDF] Total {n_normal_total} NORMAL samples → {normal_pdf_path}")

print(f"\n{'='*60}")
print(f"PLOTTING SELESAI. Output: {ML_DIR_V2}")
print(f"{'='*60}")


--- STRONG plot (combined) ---
  Cibadak: plotting 4 STRONG samples...
  Cibaliung: plotting 28 STRONG samples...
  Cileles: plotting 27 STRONG samples...
  Leuwi Dimar: plotting 23 STRONG samples...
  Singamerta: plotting 34 STRONG samples...
  Kebun Bibit Cibubur: plotting 23 STRONG samples...
  BSD Serpong: plotting 31 STRONG samples...
  Golf Modern: plotting 42 STRONG samples...
  PIK: plotting 15 STRONG samples...
  TMII: plotting 32 STRONG samples...
  Staklim Banten: plotting 16 STRONG samples...
[PDF] Total 275 STRONG samples → output_pipeline_v3\02_ml_contam_0.01\ml_strong_ALL_STATIONS.pdf

--- MEDIUM plot (combined) ---
  Cibadak: plotting 22 MEDIUM samples...
  Cibaliung: plotting 93 MEDIUM samples...
  Cileles: plotting 73 MEDIUM samples...
  Leuwi Dimar: plotting 113 MEDIUM samples...
  Singamerta: plotting 80 MEDIUM samples...
  Kebun Bibit Cibubur: plotting 83 MEDIUM samples...
  BSD Serpong: plotting 99 MEDIUM samples...
  Golf Modern: plotting 92 MEDIUM samples...
  

In [13]:
# ============================================================
# CELL TAMBAHAN: PERBANDINGAN HASIL contamination 0.001 vs 0.01
# ============================================================

print("=" * 80)
print("PERBANDINGAN HASIL ML PIPELINE 1: contamination 0.001 vs 0.01")
print("=" * 80)

comparison_rows = []

for station_name in station_files.keys():
    if station_name not in all_ml_results or station_name not in all_ml_results_v2:
        continue
    
    df_old = all_ml_results[station_name]      # contamination 0.001
    df_new = all_ml_results_v2[station_name]   # contamination 0.01
    
    comparison_rows.append({
        'station': station_name,
        'total': len(df_old),
        'old_strong': (df_old['anomaly_votes'] == 3).sum(),
        'new_strong': (df_new['anomaly_votes'] == 3).sum(),
        'old_medium': (df_old['anomaly_votes'] == 2).sum(),
        'new_medium': (df_new['anomaly_votes'] == 2).sum(),
        'old_weak':   (df_old['anomaly_votes'] == 1).sum(),
        'new_weak':   (df_new['anomaly_votes'] == 1).sum(),
        'old_total_flagged': (df_old['anomaly_votes'] >= 1).sum(),
        'new_total_flagged': (df_new['anomaly_votes'] >= 1).sum(),
    })

df_comp = pd.DataFrame(comparison_rows)
print("\n=== Per stasiun ===")
print(df_comp.to_string(index=False))

# Aggregate
old_total = df_comp['old_total_flagged'].sum()
new_total = df_comp['new_total_flagged'].sum()
total_input = df_comp['total'].sum()

print("\n=== AGGREGATE ===")
print(f"  Total ML input: {total_input}")
print(f"  Flagged dengan contamination 0.001 (vote ≥ 1): {old_total} ({100*old_total/total_input:.2f}%)")
print(f"  Flagged dengan contamination 0.01  (vote ≥ 1): {new_total} ({100*new_total/total_input:.2f}%)")
print(f"  Kenaikan: {new_total - old_total:+d} ({100*(new_total - old_total)/old_total:+.1f}%)")

# Save
comp_path = os.path.join(OUTPUT_DIR, "comparison_contamination_0.001_vs_0.01.xlsx")
df_comp.to_excel(comp_path, index=False)
print(f"\n[EXCEL] Comparison → {comp_path}")

PERBANDINGAN HASIL ML PIPELINE 1: contamination 0.001 vs 0.01

=== Per stasiun ===
            station  total  old_strong  new_strong  old_medium  new_medium  old_weak  new_weak  old_total_flagged  new_total_flagged
            Cibadak   3413           0           4           5          22        62        83                 67                109
          Cibaliung  18910           3          28          14          93       106       309                123                430
            Cileles  14603           2          27          14          73        98       221                114                321
        Leuwi Dimar  21591           1          23          17         113        91       359                109                495
         Singamerta  19225           2          34          13          80       108       320                123                434
Kebun Bibit Cibubur  18175           0          23          15          83       132       329                147      